# Faz 1 - EDA (Kesifsel Veri Analizi)

Gorev madde 11: hat bazinda gozlem sayisi, arac bazinda gozlem sayisi, saat bazinda aktif arac sayisi, eksik/duplicate/stale oranlari, hiz dagilimi, ornek arac hareketi.

Bu notebook `data/processed/` ve `data/raw/` altindaki dosyalari okur, henuz model gelistirmez (sadece EDA).

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

PROCESSED = Path("../data/processed")
RAW = Path("../data/raw")

positions = pd.read_csv(PROCESSED / "normalized_positions.csv")
ingestion_log = pd.read_csv(RAW / "ingestion_log.csv")

movement_path = PROCESSED / "movement_metrics.csv"
movement = pd.read_csv(movement_path) if movement_path.exists() else pd.DataFrame()

stale_path = PROCESSED / "stale_report.csv"
stale = pd.read_csv(stale_path) if stale_path.exists() else pd.DataFrame()

trajectory_index_path = PROCESSED / "trajectories" / "trajectory_index.csv"
trajectory_index = pd.read_csv(trajectory_index_path) if trajectory_index_path.exists() else pd.DataFrame()

print(f"normalized_positions: {len(positions)} satir")
print(f"ingestion_log: {len(ingestion_log)} satir")
print(f"movement_metrics: {len(movement)} satir")
print(f"stale_report: {len(stale)} satir")
print(f"trajectory_index: {len(trajectory_index)} satir")

normalized_positions: 1870 satir
ingestion_log: 156 satir
movement_metrics: 36 satir
stale_report: 16 satir
trajectory_index: 33 satir


## Hat bazinda GPS gozlem sayisi

In [2]:
obs_by_line = positions.groupby("line_no").size().sort_values(ascending=False)
print(obs_by_line)
obs_by_line.plot(kind="bar", title="Hat bazinda GPS gozlem sayisi")
plt.ylabel("gozlem sayisi")
plt.tight_layout()
plt.show()

line_no
515    1229
121     428
761     213
dtype: int64


RecursionError: maximum recursion depth exceeded

Error in callback <function _draw_all_if_interactive at 0x0000029E6C7CEAE0> (for post_execute), with arguments args (),kwargs {}:


RecursionError: maximum recursion depth exceeded

RecursionError: maximum recursion depth exceeded

<Figure size 640x480 with 1 Axes>

## Arac bazinda gozlem sayisi (ilk 20)

In [ ]:
obs_by_vehicle = positions.groupby(["line_no", "vehicle_id"]).size().sort_values(ascending=False)
obs_by_vehicle.head(20)

## Saat bazinda aktif arac sayisi

In [ ]:
positions["observed_at_dt"] = pd.to_datetime(positions["observed_at"])
positions["hour"] = positions["observed_at_dt"].dt.floor("h")

active_by_hour = positions.groupby("hour")["vehicle_id"].nunique()
active_by_hour.plot(kind="line", marker="o", title="Saat bazinda benzersiz aktif arac sayisi")
plt.ylabel("benzersiz arac sayisi")
plt.tight_layout()
plt.show()

## Eksik / gecersiz GPS orani (is_valid = False)

In [ ]:
invalid_rate = (positions["is_valid"] == False).mean()
print(f"Toplam gecersiz gozlem orani: {invalid_rate:.2%}")

flag_counts = positions.loc[positions["is_valid"] == False, "quality_flags"].value_counts()
print("\nFlag dagilimi:")
print(flag_counts)

## Duplicate orani (ingestion_log uzerinden)

In [ ]:
if len(ingestion_log) > 0 and "duplicate_count" in ingestion_log.columns:
    total_vehicles = ingestion_log["vehicle_count"].sum()
    total_duplicates = ingestion_log["duplicate_count"].sum()
    dup_rate = total_duplicates / total_vehicles if total_vehicles > 0 else 0
    print(f"Toplam arac kaydi: {total_vehicles}")
    print(f"Toplam gercek duplicate: {total_duplicates}")
    print(f"Duplicate orani: {dup_rate:.2%}")

## Stale (uzun sure hareketsiz) GPS orani

In [ ]:
if len(stale) > 0:
    print(f"Tespit edilen stale seri sayisi: {len(stale)}")
    print(stale[["line_no", "vehicle_id", "run_length"]].sort_values("run_length", ascending=False))
else:
    print("Stale seri bulunamadi (esik: 3+ ardisik ayni konum)")

## Hesaplanan hiz dagilimi

In [ ]:
if len(movement) > 0:
    speeds = movement["calculated_speed_kmh"].dropna()
    print(speeds.describe())
    speeds.plot(kind="hist", bins=20, title="Hesaplanan hiz dagilimi (km/h)")
    plt.xlabel("km/h")
    plt.tight_layout()
    plt.show()

    unrealistic_rate = movement["is_unrealistic_speed"].mean()
    print(f"\nGercekci olmayan hiz orani: {unrealistic_rate:.2%}")
else:
    print("movement_metrics.csv henuz yok - once 'python scripts/compute_movement_metrics.py' calistirin")

## Ornek: bir aracin haritada hareketi

In [ ]:
import folium

if len(trajectory_index) > 0:
    # en cok noktasi olan trajectory'yi ornek olarak sec
    example = trajectory_index.sort_values("point_count", ascending=False).iloc[0]
    print(f"Ornek: Hat {example['line_no']}, Arac {example['vehicle_id']}, {example['point_count']} nokta")

    import json
    with open(example["file_path"], encoding="utf-8") as f:
        traj = json.load(f)

    points = [(p["latitude"], p["longitude"]) for p in traj["points"]]
    center = points[len(points) // 2]

    m = folium.Map(location=center, zoom_start=14)
    folium.PolyLine(points, color="blue", weight=3).add_to(m)
    for i, (lat, lon) in enumerate(points):
        folium.CircleMarker(
            location=(lat, lon), radius=4,
            color="green" if i == 0 else ("red" if i == len(points) - 1 else "blue"),
            popup=f"Nokta {i}"
        ).add_to(m)
    m
else:
    print("trajectory_index.csv henuz yok - once 'python app/trajectory/trajectory_builder.py' calistirin")

## Ozet

Bu notebook, collector calistikca (`python scripts/run_collector.py`) biriken veriyle tekrar tekrar calistirilabilir. Guncel sonuc icin: Kernel > Restart & Run All.